# Registro histórico: 03_mobilenet_training

Origem: `C02` — `Colab Notebooks/MobileNetTraining_01.ipynb`.

MobileNetV3-Large com duas saídas e fine-tuning parcial.

**Limitações conhecidas:** BCELoss incompatível com logits de duas saídas e alvos escalares; não há treino completo salvo. Não usar como reprodução do checkpoint avaliado em Summary.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


# 1. Imports e setups



In [ ]:
import os
import random
import shutil
from pathlib import Path
from timeit import default_timer as timer

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import requests

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
from torchvision import transforms, datasets
from torchvision.models import MobileNet_V3_Large_Weights

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
# Set random seeds for reproducibility
random.seed(91432)
np.random.seed(91432)
torch.manual_seed(91432)
if torch.cuda.is_available():
    torch.cuda.manual_seed(91432)

# 2. Importando o Dataset do Google Drive

In [ ]:
if not os.path.exists("/content/0-vgg"):
    print("Importando dataset...")
    from google.colab import drive
    drive.mount('/content/drive')

    # Copy dataset from Drive to Colab
    !cp -r "/content/drive/MyDrive/0-vgg" "/content/"
else:
    print("Dataset já importado")

# Setup paths
image_path = Path("/content/0-vgg/base_dir")
train_dir = image_path / "train_dir"
test_dir = image_path / "test_dir"
val_dir = image_path / "val_dir"

print(f"Dataset path: {image_path}")
print(f"Train directory: {train_dir}")
print(f"Test directory: {test_dir}")
print(f"Validation directory: {val_dir}")


#3. Download Helper Functions

In [ ]:
# Download helper functions from Learn PyTorch repo
if Path("helper_functions.py").is_file():
    print("helper_functions.py already exists, skipping download")
else:
    print("Downloading helper_functions.py")
    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
    with open("helper_functions.py", "wb") as f:
        f.write(request.content)

from helper_functions import accuracy_fn

# 4. Visualização e levantamento do dataset

In [ ]:
def count_images_per_class(directory):
    """Counts the number of image files per class in a directory."""
    class_counts = {}
    for class_name in os.listdir(directory):
        class_dir = os.path.join(directory, class_name)
        if os.path.isdir(class_dir):
            image_count = 0
            for _, _, files in os.walk(class_dir):
                for file in files:
                    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                        image_count += 1
            class_counts[class_name] = image_count
    return class_counts

# Count images per class in each directory
train_class_counts = count_images_per_class(train_dir)
test_class_counts = count_images_per_class(test_dir)
val_class_counts = count_images_per_class(val_dir)

print("Number of images per class:")
print(f"  Train directory: {train_class_counts}")
print(f"  Test directory: {test_class_counts}")
print(f"  Validation directory: {val_class_counts}")

# Get all image paths
image_path_list = list(image_path.glob("*/*/*.jpg"))
print(f"\nTotal images found: {len(image_path_list)}")

# Display a random image
random_image_path = random.choice(image_path_list)
image_class = random_image_path.parent.stem
print(f"\nRandom image path: {random_image_path}")
print(f"Image class: {image_class}")

img = Image.open(random_image_path)
print(f"Image dimensions: {img.width}x{img.height}")
print(f"Image mode: {img.mode}")

plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.title(f"Sample Image - Class: {image_class}", fontsize=14)
plt.axis("off")
plt.show()

# 5. Data transforms para a arquitetura MobileNetV3

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize to slightly larger first
    transforms.RandomCrop(224),     # Random crop for data augmentation

    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                        std=[0.229, 0.224, 0.225])
])

# Test/validation transforms (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

print("Data transforms created successfully!")

# 6. Visualização das imagens Transformadas

In [ ]:
def plot_transformed_images(image_paths, transform, n=3, seed=42):
    """Plot original and transformed images side by side."""
    if seed:
        random.seed(seed)
    random_image_paths = random.sample(image_paths, k=n)

    fig, axes = plt.subplots(n, 2, figsize=(10, 4*n))
    if n == 1:
        axes = axes.reshape(1, -1)

    for i, image_path in enumerate(random_image_paths):
        with Image.open(image_path) as f:
            # Original image
            axes[i, 0].imshow(f)
            axes[i, 0].set_title("Original")
            axes[i, 0].axis("off")

            # Transformed image
            transformed_image = transform(f).permute(1, 2, 0)
            # Denormalize for visualization
            mean = torch.tensor([0.485, 0.456, 0.406])
            std = torch.tensor([0.229, 0.224, 0.225])
            transformed_image = transformed_image * std + mean
            transformed_image = torch.clamp(transformed_image, 0, 1)

            axes[i, 1].imshow(transformed_image)
            axes[i, 1].set_title("Transformed")
            axes[i, 1].axis("off")

            fig.suptitle(f"Class: {image_path.parent.stem}", fontsize=16)

    plt.tight_layout()
    plt.show()

plot_transformed_images(image_paths=image_path_list,
                       transform=train_transform,
                       n=2,
                       seed=42)

# 7. Criando os Dataloaders

In [ ]:
# Create datasets
train_data = datasets.ImageFolder(root=train_dir, transform=train_transform)
test_data = datasets.ImageFolder(root=test_dir, transform=test_transform)
val_data = datasets.ImageFolder(root=val_dir, transform=test_transform)

# Get class information
class_names = train_data.classes
class_dict = train_data.class_to_idx
print(f"Classes: {class_names}")
print(f"Class to index mapping: {class_dict}")
print(f"Dataset sizes - Train: {len(train_data)}, Test: {len(test_data)}, Val: {len(val_data)}")

# Optimized batch size for MobileNetV3
BATCH_SIZE = 32  # Smaller batch size for better convergence

# Create dataloaders
train_dataloader = DataLoader(dataset=train_data,
                             batch_size=BATCH_SIZE,
                             shuffle=True,
                             num_workers=2,  # For faster data loading
                             pin_memory=True if device == "cuda" else False)

val_dataloader = DataLoader(dataset=val_data,
                            batch_size=BATCH_SIZE,
                            shuffle=False,
                            num_workers=2,
                            pin_memory=True if device == "cuda" else False)

test_dataloader = DataLoader(dataset=test_data,
                           batch_size=BATCH_SIZE,
                           shuffle=False,
                           num_workers=2,
                           pin_memory=True if device == "cuda" else False)

print(f"DataLoaders created - Train batches: {len(train_dataloader)}, Test batches: {len(test_dataloader)}, Val batches: {len(val_dataloader)}")

# 8. Criando o Modelo MobileNetV3

In [ ]:
def create_mobilenetv3_model(num_classes=2, pretrained=True):
    """Create and modify MobileNetV3 for binary classification."""
    # Load pretrained MobileNetV3-Large
    if pretrained:
        weights = MobileNet_V3_Large_Weights.IMAGENET1K_V1
        model = models.mobilenet_v3_large(weights=weights)
        print("Loaded MobileNetV3-Large with ImageNet pretrained weights")
    else:
        model = models.mobilenet_v3_large(weights=None)
        print("Created MobileNetV3-Large without pretrained weights")

    # Freeze early layers for transfer learning
    for param in model.features[:-3].parameters():  # Freeze all but last 3 feature layers
        param.requires_grad = False

    # Modify classifier for binary classification
    num_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(num_features, num_classes)

    print(f"Modified classifier for {num_classes} classes")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    return model

# Create model
model = create_mobilenetv3_model(num_classes=2, pretrained=True)
model = model.to(device)

# 9. Setup do treinamento

In [ ]:
# Loss function and optimizer optimized for MobileNetV3
loss_fn = nn.BCELoss()

# Using AdamW optimizer with weight decay for better generalization
optimizer = optim.AdamW(model.parameters(),
                       lr=1e-4,  # Lower learning rate for fine-tuning
                       weight_decay=1e-4)  # Weight decay for regularization

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                               mode='min',
                                               factor=0.5,
                                               patience=3)

print("Training configuration setup complete!")
print(f"Loss function: {loss_fn}")
print(f"Optimizer: {optimizer}")
print(f"Learning rate scheduler: {scheduler}")

# 10. Funções de treino e validação

In [ ]:
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device = device):
    """Enhanced training step with better progress tracking."""
    model.train()
    train_loss, train_acc = 0, 0

    # Progress bar with better description
    progress_bar = tqdm(data_loader, desc="Training", leave=False)

    for batch, (X, y) in enumerate(progress_bar):
        # Send data to device
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

        # Forward pass
        y_pred = model(X)
        loss = loss_fn(y_pred, y)

        # Calculate accuracy
        acc = accuracy_fn(y_true=y, y_pred=y_pred.argmax(dim=1))

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update metrics
        train_loss += loss.item()
        train_acc += acc

        # Update progress bar
        progress_bar.set_postfix({
            'batch_loss': f'{loss.item():.4f}',
            'batch_acc': f'{acc:.2f}%'
        })

    # Calculate average metrics
    train_loss /= len(data_loader)
    train_acc /= len(data_loader)

    return train_loss, train_acc

def val_step(model: torch.nn.Module,
             data_loader: torch.utils.data.DataLoader,
             loss_fn: torch.nn.Module,
             device: torch.device = device):
    """Enhanced validation step with better progress tracking."""
    model.eval()
    val_loss, val_acc = 0, 0

    # Progress bar with better description
    progress_bar = tqdm(data_loader, desc="Validation", leave=False)

    with torch.inference_mode():
        for X, y in progress_bar:
            # Send data to device
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

            # Forward pass
            y_pred = model(X)
            loss = loss_fn(y_pred, y)
            acc = accuracy_fn(y_true=y, y_pred=y_pred.argmax(dim=1))

            # Update metrics
            val_loss += loss.item()
            val_acc += acc

            # Update progress bar
            progress_bar.set_postfix({
                'batch_loss': f'{loss.item():.4f}',
                'batch_acc': f'{acc:.2f}%'
            })

    # Calculate average metrics
    val_loss /= len(data_loader)
    val_acc /= len(data_loader)

    return val_loss, val_acc

def print_train_time(start: float, end: float, device: torch.device = None):
    """Print training time."""
    total_time = end - start
    print(f"Train time on {device}: {total_time:.3f} seconds")
    return total_time

# 11. Loop de treino

In [ ]:
# Training configuration
EPOCHS = 50  # More epochs since we'll use early stopping
PATIENCE = 7  # Early stopping patience
MIN_DELTA = 0.01  # Minimum improvement threshold

best_val_acc = 0.0
best_model_state = None
epochs_no_improve = 0

# Training history
train_losses, train_accs = [], []
val_losses, val_accs = [], []

print("Starting training with Early Stopping...")
print(f"Max epochs: {EPOCHS} | Patience: {PATIENCE} | Min delta: {MIN_DELTA}")
print("-" * 50)

# Start training timer
train_start_time = timer()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 30)

    # Training phase
    train_loss, train_acc = train_step(
        model=model,
        data_loader=train_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device
    )

    # Validation phase
    val_loss, val_acc = val_step(
        model=model,
        data_loader=val_dataloader,
        loss_fn=loss_fn,
        device=device
    )

    # Learning rate scheduling
    scheduler.step(val_loss)

    # Store metrics
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Print epoch results
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")

    # Early Stopping Logic
    if val_acc > best_val_acc + MIN_DELTA:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        epochs_no_improve = 0
        print(f"🎉 New best validation accuracy: {best_val_acc:.2f}%")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement for {epochs_no_improve}/{PATIENCE} epochs")

        if epochs_no_improve >= PATIENCE:
            print(f"🛑 Early stopping triggered after {epoch+1} epochs")
            print(f"📊 Best validation accuracy: {best_val_acc:.2f}%")
            break

# End training timer
train_end_time = timer()
total_train_time = print_train_time(train_start_time, train_end_time, device)

print(f"\nTraining completed!")
print(f"Best validation accuracy: {best_val_acc:.2f}%")

# 12. Plot do histórico do treinamento

In [ ]:
def plot_training_history(train_losses, val_losses, train_accs, val_accs):
    """Plot training and validation metrics."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Plot losses
    epochs_range = range(1, len(train_losses) + 1)
    ax1.plot(epochs_range, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs_range, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot accuracies
    ax2.plot(epochs_range, train_accs, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs_range, val_accs, 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_training_history(train_losses, val_losses, train_accs, val_accs)


# 13. Evaluation do modelo e testes

In [ ]:
# Load best model and evaluate
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("Loaded best model for evaluation")

# Evaluate on test set
print("Evaluating model on test set...")
test_results = eval_model(model=model,
                         data_loader=test_dataloader,
                         loss_fn=loss_fn,
                         device=device)

print(f"\n🎯 Final Test Results:")
print(f"Model: {test_results['model_name']}")
print(f"Test Loss: {test_results['model_loss']:.4f}")
print(f"Test Accuracy: {test_results['model_acc']:.2f}%")

# 14. Salvando o Modelo

In [ ]:
# Define the base path for local saving (if still needed, otherwise can be removed)
LOCAL_MODEL_PATH = Path("/content/0-vgg/models")
LOCAL_MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Define the base path in Google Drive
DRIVE_MODEL_PATH = Path("/content/drive/MyDrive/0-vgg/models")
DRIVE_MODEL_PATH.mkdir(parents=True, exist_ok=True)  # Create directory if it doesn't exist

# Save model
MODEL_NAME = "MobileNetV3_SkinLesion_Binary_Classifier_1.pth"

# --- Logic for local model saving ---
MODEL_SAVE_PATH_LOCAL = LOCAL_MODEL_PATH / MODEL_NAME

# Check if local model file already exists and append a number if necessary
counter = 1
original_model_name_local = MODEL_SAVE_PATH_LOCAL.stem
while MODEL_SAVE_PATH_LOCAL.exists():
    MODEL_SAVE_PATH_LOCAL = LOCAL_MODEL_PATH / f"{original_model_name_local}_{counter}{MODEL_SAVE_PATH_LOCAL.suffix}"
    counter += 1

print(f"Saving model locally to: {MODEL_SAVE_PATH_LOCAL}")
torch.save(obj=model.state_dict(), f=MODEL_SAVE_PATH_LOCAL)

# --- Logic for Google Drive model saving ---
MODEL_SAVE_PATH_DRIVE = DRIVE_MODEL_PATH / MODEL_NAME

# Check if Drive model file already exists and append a number if necessary
counter = 1
original_model_name_drive = MODEL_SAVE_PATH_DRIVE.stem
while MODEL_SAVE_PATH_DRIVE.exists():
    MODEL_SAVE_PATH_DRIVE = DRIVE_MODEL_PATH / f"{original_model_name_drive}_{counter}{MODEL_SAVE_PATH_DRIVE.suffix}"
    counter += 1

print(f"Saving model to Google Drive: {MODEL_SAVE_PATH_DRIVE}")
torch.save(obj=model.state_dict(), f=MODEL_SAVE_PATH_DRIVE)

# Save training history
import json
history = {
    "train_losses": train_losses,
    "val_losses": val_losses,
    "train_accs": train_accs,
    "val_accs": val_accs,
    "test_results": {
        "loss": test_results['model_loss'],
        "accuracy": test_results['model_acc']
    },
    "training_time": total_train_time,
    "epochs": EPOCHS,
    "best_val_acc": best_val_acc
}

# Define the path for the training history JSON in Google Drive
HISTORY_NAME = "training_history_MobileNetV3_1.json"
HISTORY_SAVE_PATH_DRIVE = DRIVE_MODEL_PATH / HISTORY_NAME

# Check if history file already exists and append a number if necessary
counter = 1
original_history_name = HISTORY_SAVE_PATH_DRIVE.stem
while HISTORY_SAVE_PATH_DRIVE.exists():
    HISTORY_SAVE_PATH_DRIVE = DRIVE_MODEL_PATH / f"{original_history_name}_{counter}{HISTORY_SAVE_PATH_DRIVE.suffix}"
    counter += 1

with open(HISTORY_SAVE_PATH_DRIVE, "w") as f:
    json.dump(history, f, indent=2)

print("Model and training history saved successfully to Google Drive!")

# 15. Fazendo Previsões

In [ ]:
def make_predictions(model: torch.nn.Module,
                    data: list,
                    device: torch.device = device):
    """Make predictions on a list of samples."""
    pred_probs = []
    model.eval()

    with torch.inference_mode():
        for sample in data:
            # Prepare sample
            sample = torch.unsqueeze(sample, dim=0).to(device)

            # Make prediction
            pred_logit = model(sample)
            pred_prob = torch.softmax(pred_logit.squeeze(), dim=0)
            pred_probs.append(pred_prob.cpu())

    return torch.stack(pred_probs)

def visualize_predictions(model, dataset, class_names, num_samples=6):
    """Visualize model predictions on random samples."""
    # Get random samples
    test_samples = []
    test_labels = []

    for sample, label in random.sample(list(dataset), k=num_samples):
        test_samples.append(sample)
        test_labels.append(label)

    # Make predictions
    pred_probs = make_predictions(model=model, data=test_samples)
    pred_classes = pred_probs.argmax(dim=1)

    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()

    for i, sample in enumerate(test_samples):
        # Denormalize image for display
        img = sample.permute(1, 2, 0)
        mean = torch.tensor([0.485, 0.456, 0.406])
        std = torch.tensor([0.229, 0.224, 0.225])
        img = img * std + mean
        img = torch.clamp(img, 0, 1)

        axes[i].imshow(img)

        pred_label = class_names[pred_classes[i]]
        truth_label = class_names[test_labels[i]]
        confidence = pred_probs[i].max().item() * 100

        title_text = f"Pred: {pred_label} ({confidence:.1f}%)\nTruth: {truth_label}"

        if pred_label == truth_label:
            axes[i].set_title(title_text, fontsize=12, color="green", fontweight='bold')
        else:
            axes[i].set_title(title_text, fontsize=12, color="red", fontweight='bold')

        axes[i].axis("off")

    plt.tight_layout()
    plt.suptitle("Model Predictions on Test Samples", fontsize=16, fontweight='bold', y=1.02)
    plt.show()

# Visualize predictions
print("Visualizing model predictions...")
visualize_predictions(model, val_data, class_names, num_samples=6)

 # 16. Sumário e Resultados

In [ ]:
print("=" * 60)
print("📊 TRAINING SUMMARY")
print("=" * 60)
print(f"Model: MobileNetV3-Large")
print(f"Task: Binary Classification (Melanoma vs Non-Melanoma)")
print(f"Dataset Size: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")
print(f"Training Time: {total_train_time:.2f} seconds")
print(f"Epochs Trained: {EPOCHS}")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"Final Test Accuracy: {test_results['model_acc']:.2f}%")
print(f"Model Saved: {MODEL_SAVE_PATH_LOCAL}")
print("=" * 60)

# Performance metrics
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Generate classification report
y_true = test_results['labels']
y_pred = test_results['predictions']

print("\n📈 Detailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Plot confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
def count_images_per_class(directory):
    """Counts the number of image files per class in a directory."""
    class_counts = {}
    for class_name in os.listdir(directory):
        class_dir = os.path.join(directory, class_name)
        if os.path.isdir(class_dir):
            image_count = 0
            for _, _, files in os.walk(class_dir):
                for file in files:
                    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                        image_count += 1
            class_counts[class_name] = image_count
    return class_counts

# Count images per class in each directory
train_class_counts = count_images_per_class(train_dir)
test_class_counts = count_images_per_class(test_dir)
val_class_counts = count_images_per_class(val_dir)

print("Number of images per class:")
print(f"  Train directory: {train_class_counts}")
print(f"  Test directory: {test_class_counts}")
print(f"  Validation directory: {val_class_counts}")

# Get all image paths
image_path_list = list(image_path.glob("*/*/*.jpg"))
print(f"\nTotal images found: {len(image_path_list)}")

# Display a random image
random_image_path = random.choice(image_path_list)
image_class = random_image_path.parent.stem
print(f"\nRandom image path: {random_image_path}")
print(f"Image class: {image_class}")

img = Image.open(random_image_path)
print(f"Image dimensions: {img.width}x{img.height}")
print(f"Image mode: {img.mode}")

plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.title(f"Sample Image - Class: {image_class}", fontsize=14)
plt.axis("off")
plt.show()